# Notebook 06 — Performance Evaluation

**Input:** checkpoint from nb05, test splits from nb01, mol representations from nb03  
**GPU:** optional (inference only)

| Section | Description |
|---------|-------------|
| 1 | Load model + run inference on held-out test (once, final) |
| 2 | Youden threshold from train-pool CV |
| 3 | Full metrics with 95% bootstrap CI (N=1000) |
| 4 | ROC & PR curves — 6-panel |
| 5 | Scaffold CV vs Random CV comparison |
| 6 | Baseline comparison (RF, SVM, XGBoost, single-modality NNs) |
| 7 | AD-vs-performance (Tanimoto-stratified AUROC) |

---
## 0 — Imports & Config

In [ ]:
import sys, os, gc, json, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import torch
import yaml
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.metrics import (
    roc_auc_score, average_precision_score, matthews_corrcoef,
    balanced_accuracy_score, f1_score, confusion_matrix,
    roc_curve, precision_recall_curve
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem

warnings.filterwarnings('ignore')

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))

with open(REPO_ROOT / 'config.yaml') as f:
    CFG = yaml.safe_load(f)

SEED         = CFG['seed']
SPLIT_ROOT   = Path(CFG['paths']['split_root'])
MOL_REP_ROOT = Path(CFG['paths']['mol_rep_root'])
TRAIN_ROOT   = Path(CFG['paths']['training_root'])
RESULT_DIR   = Path(CFG['paths']['results_root']) / 'evaluation'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

TASKS   = [t['name'] for t in CFG['targets']]
N_TASKS = len(TASKS)
SHORT   = {t['name']: t['label'] for t in CFG['targets']}
DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
N_BOOT  = 1000

print(f'Device    : {DEVICE}')
print(f'RESULT_DIR: {RESULT_DIR}')

---
## 1 — Load Model & Run Inference on Held-out Test

> This is the **single, final evaluation**. The held-out test set was not touched during training or Optuna search.

In [ ]:
# Re-import model class from nb05 (copy the class definitions here or import from a shared module)
# For brevity we load from checkpoint and assume MultiModalQSAR is importable
# If running standalone, paste the full class definitions from nb05 Section 1 here.

ckpt_path = TRAIN_ROOT / 'final_retrain_last.pt'
ckpt      = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
best_config = ckpt['config']
MORD_DIM    = ckpt['mord_dim']
print(f'Config: {json.dumps(best_config, indent=2)}')
print(f'Mordred dim: {MORD_DIM}')

# --- Paste or import MultiModalQSAR here ---
# from notebooks.model import MultiModalQSAR  # if you refactor into a module
# OR copy the class directly from nb05.
# We assume `model` is available after running this cell.
# model = MultiModalQSAR(best_config, MORD_DIM).to(DEVICE)
# model.load_state_dict(ckpt['model_state'])
# model.eval()
# print('✅ Model loaded.')

In [ ]:
# Load protein tensor
prot_dict      = torch.load(MOL_REP_ROOT / 'protein_embeddings' / 'all_proteins_esm2_mean.pt',
                             map_location='cpu')
protein_tensor = torch.stack([prot_dict[t] for t in TASKS]).to(DEVICE)

# ── Run inference (replace with your DataLoader from nb05) ───────────────────
# Assumes test_loader is available; if not, reconstruct it using the same
# MultiModalDataset + collate_fn as in nb05, using split='test' files.

@torch.no_grad()
def run_inference(model, loader):
    model.eval()
    all_preds, all_labels, all_masks = [], [], []
    for batch in loader:
        batch  = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k,v in batch.items()}
        logits, _, _ = model(batch, protein_tensor)
        all_preds.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(batch['labels'].cpu().numpy())
        all_masks.append(batch['masks'].cpu().numpy())
    return np.vstack(all_preds), np.vstack(all_labels), np.vstack(all_masks)

# preds, labels, masks = run_inference(model, test_loader)
# print(f'Inference done: {preds.shape} predictions')

# ── OR: load pre-saved predictions if inference was already run ───────────────
pred_path = RESULT_DIR / 'test_predictions.npz'
if pred_path.exists():
    data   = np.load(pred_path)
    preds  = data['preds']
    labels = data['labels']
    masks  = data['masks']
    print(f'Loaded saved predictions: {preds.shape}')
else:
    print('Run inference first and save: np.savez(pred_path, preds=preds, labels=labels, masks=masks)')

---
## 2 — Youden-Optimal Threshold (from Train Pool CV)

Threshold is determined on train-pool CV folds only, then **fixed** and applied once to the test set.  
This prevents threshold leakage from the held-out test.

In [ ]:
def youden_threshold(y_true, y_prob):
    """Find threshold maximising Youden index (sensitivity + specificity - 1)."""
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    j_scores = tpr - fpr
    return float(thresholds[np.argmax(j_scores)])


# Load train predictions from CV (saved during nb05 if available)
# Otherwise default to 0.5
cv_pred_path = TRAIN_ROOT / 'cv_val_predictions.npz'

if cv_pred_path.exists():
    cv_data    = np.load(cv_pred_path)
    cv_preds   = cv_data['preds']
    cv_labels  = cv_data['labels']
    cv_masks   = cv_data['masks']
    youden_thr = {}
    for i, task in enumerate(TASKS):
        m = cv_masks[:, i].astype(bool)
        if m.sum() > 0 and len(np.unique(cv_labels[m, i])) > 1:
            youden_thr[task] = youden_threshold(cv_labels[m, i], cv_preds[m, i])
        else:
            youden_thr[task] = 0.5
    print('Youden thresholds (from CV):')
else:
    youden_thr = {task: 0.5 for task in TASKS}
    print('CV predictions not found — using default threshold 0.5 for all tasks.')

for task, thr in youden_thr.items():
    print(f'  {task:<22}: {thr:.4f}')

---
## 3 — Full Metrics + 95% Bootstrap CI

Non-parametric bootstrap (N=1000). Threshold-dependent metrics use Youden threshold.

In [ ]:
def compute_all_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    sens = tp / max(tp + fn, 1)
    spec = tn / max(tn + fp, 1)
    return {
        'AUROC':    roc_auc_score(y_true, y_prob),
        'AUPRC':    average_precision_score(y_true, y_prob),
        'MCC':      matthews_corrcoef(y_true, y_pred),
        'BalAcc':   balanced_accuracy_score(y_true, y_pred),
        'F1':       f1_score(y_true, y_pred, zero_division=0),
        'Sens':     sens,
        'Spec':     spec,
    }

def bootstrap_ci(y_true, y_prob, threshold=0.5, n_boot=N_BOOT, seed=SEED):
    rng   = np.random.RandomState(seed)
    point = compute_all_metrics(y_true, y_prob, threshold)
    boots = {k: [] for k in point}
    for _ in range(n_boot):
        idx = rng.choice(len(y_true), len(y_true), replace=True)
        yt, yp = y_true[idx], y_prob[idx]
        if len(np.unique(yt)) < 2: continue
        m = compute_all_metrics(yt, yp, threshold)
        for k, v in m.items(): boots[k].append(v)
    ci = {}
    for k, pt in point.items():
        lo, hi = np.percentile(boots[k], [2.5, 97.5]) if boots[k] else (np.nan, np.nan)
        ci[k] = {'point': pt, 'lo': lo, 'hi': hi,
                 'fmt': f'{pt:.3f} [{lo:.3f}–{hi:.3f}]'}
    return ci


results = {}
print(f'{"Task":<22} {"AUROC":>22} {"AUPRC":>22} {"MCC":>22}')
print('-' * 90)

for i, task in enumerate(TASKS):
    m  = masks[:, i].astype(bool)
    yt = labels[m, i]
    yp = preds[m, i]
    if len(np.unique(yt)) < 2:
        print(f'  {task:<22} — skipped (single class)')
        continue
    thr = youden_thr.get(task, 0.5)
    ci  = bootstrap_ci(yt, yp, threshold=thr)
    results[task] = {'ci': ci, 'y_true': yt, 'y_prob': yp, 'threshold': thr}
    print(f'  {task:<22} {ci["AUROC"]["fmt"]:>22} {ci["AUPRC"]["fmt"]:>22} {ci["MCC"]["fmt"]:>22}')

# Mean across tasks
mean_auroc = np.mean([results[t]['ci']['AUROC']['point'] for t in results])
print(f'\n  Mean AUROC (all tasks): {mean_auroc:.4f}')

In [ ]:
# ── Table 3: full metrics per endpoint ───────────────────────────────────────
rows = []
for task, res in results.items():
    row = {'Endpoint': SHORT[task]}
    for metric in ['AUROC','AUPRC','MCC','BalAcc','F1','Sens','Spec']:
        row[metric] = res['ci'][metric]['fmt']
    rows.append(row)

table3 = pd.DataFrame(rows).set_index('Endpoint')
print('\nTABLE 3 — Main model performance (held-out test, 95% bootstrap CI)')
print(table3.to_string())
table3.to_csv(RESULT_DIR / 'table3_main_performance.csv')
print(f'\nSaved → {RESULT_DIR}/table3_main_performance.csv')

---
## 4 — ROC & Precision-Recall Curves

In [ ]:
fig, axes = plt.subplots(2, N_TASKS, figsize=(4*N_TASKS, 8))
COLOR = '#3B8BD4'

for j, task in enumerate(TASKS):
    if task not in results: continue
    yt = results[task]['y_true']
    yp = results[task]['y_prob']
    thr = results[task]['threshold']
    auroc = results[task]['ci']['AUROC']['point']
    auprc = results[task]['ci']['AUPRC']['point']

    # ROC
    fpr, tpr, _ = roc_curve(yt, yp)
    ax = axes[0][j]
    ax.plot(fpr, tpr, color=COLOR, lw=1.5, label=f'AUROC={auroc:.3f}')
    ax.plot([0,1],[0,1], 'k--', lw=0.8, alpha=0.5)
    ax.set_title(SHORT[task], fontsize=10, fontweight='bold')
    ax.set_xlabel('FPR', fontsize=8); ax.set_ylabel('TPR', fontsize=8)
    ax.legend(fontsize=8, loc='lower right'); ax.tick_params(labelsize=7)

    # PR
    prec, rec, _ = precision_recall_curve(yt, yp)
    baseline = yt.mean()
    ax2 = axes[1][j]
    ax2.plot(rec, prec, color='#E8593C', lw=1.5, label=f'AUPRC={auprc:.3f}')
    ax2.axhline(baseline, color='k', ls='--', lw=0.8, alpha=0.5)
    ax2.set_xlabel('Recall', fontsize=8); ax2.set_ylabel('Precision', fontsize=8)
    ax2.legend(fontsize=8, loc='upper right'); ax2.tick_params(labelsize=7)

axes[0][0].set_ylabel('TPR', fontsize=9)
axes[1][0].set_ylabel('Precision', fontsize=9)
fig.suptitle('ROC Curves (top) and Precision-Recall Curves (bottom) — Held-out Test',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULT_DIR / 'figure3_roc_pr_curves.png', dpi=200, bbox_inches='tight')
plt.savefig(RESULT_DIR / 'figure3_roc_pr_curves.pdf', bbox_inches='tight')
plt.show()

---
## 5 — Scaffold CV vs Random CV Comparison

Shows how scaffold split gives a more conservative (realistic) performance estimate than random split — key methodological point of the paper.

In [ ]:
# Load CV AUROC logs saved during nb05 (scaffold CV and random CV)
scaffold_cv_path = TRAIN_ROOT / 'scaffold_cv_aurocs.json'
random_cv_path   = TRAIN_ROOT / 'random_cv_aurocs.json'

if scaffold_cv_path.exists() and random_cv_path.exists():
    with open(scaffold_cv_path) as f: scaffold_cv = json.load(f)  # {task: [fold_aurocs]}
    with open(random_cv_path)   as f: random_cv   = json.load(f)

    rows = []
    for task in TASKS:
        sc = scaffold_cv.get(task, [])
        rc = random_cv.get(task, [])
        rows.append({
            'Endpoint':      SHORT[task],
            'Scaffold CV':   f'{np.mean(sc):.3f} ± {np.std(sc):.3f}' if sc else 'N/A',
            'Random CV':     f'{np.mean(rc):.3f} ± {np.std(rc):.3f}' if rc else 'N/A',
            'Δ (Scaffold−Random)': f'{np.mean(sc)-np.mean(rc):+.3f}' if sc and rc else 'N/A',
        })

    cv_compare = pd.DataFrame(rows).set_index('Endpoint')
    print('Scaffold CV vs Random CV — Mean AUROC (10-fold)')
    print(cv_compare.to_string())
    cv_compare.to_csv(RESULT_DIR / 'cv_scaffold_vs_random.csv')
else:
    print('CV log files not found. Save fold-level AUROC dicts during nb05 training loop.')
    print('Expected: scaffold_cv_aurocs.json and random_cv_aurocs.json in training_root.')

---
## 6 — Baseline Comparison

Classical ML (RF, SVM, XGBoost on ECFP4) trained and evaluated on the **same splits**.  
All baselines are single-task per endpoint.

In [ ]:
def get_ecfp4(smiles_series, nbits=2048):
    X = []
    for smi in smiles_series:
        mol = Chem.MolFromSmiles(str(smi))
        if mol:
            fp  = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nbits)
            arr = np.zeros(nbits, dtype=np.uint8)
            DataStructs.ConvertToNumpyArray(fp, arr)
            X.append(arr)
        else:
            X.append(np.zeros(nbits, dtype=np.uint8))
    return np.array(X)


BASELINES = {
    'RF':      RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1),
    'SVM':     SVC(kernel='rbf', probability=True, random_state=SEED),
    'XGBoost': XGBClassifier(n_estimators=200, random_state=SEED,
                              eval_metric='logloss', verbosity=0),
}

smi_col = 'canonical_smiles'
baseline_results = {name: {} for name in BASELINES}

for task in TASKS:
    tr_path = SPLIT_ROOT / task / 'train.xlsx'
    te_path = SPLIT_ROOT / task / 'test.xlsx'
    if not tr_path.exists(): continue

    tr_df = pd.read_excel(tr_path)
    te_df = pd.read_excel(te_path)
    sc    = smi_col if smi_col in tr_df.columns else 'Smiles'

    X_tr = get_ecfp4(tr_df[sc])
    y_tr = tr_df['label'].values
    X_te = get_ecfp4(te_df[sc])
    y_te = te_df['label'].values

    for name, clf in BASELINES.items():
        import copy
        clf_fit = copy.deepcopy(clf)
        clf_fit.fit(X_tr, y_tr)
        y_prob  = clf_fit.predict_proba(X_te)[:, 1]
        if len(np.unique(y_te)) < 2: continue
        baseline_results[name][task] = {
            'AUROC': roc_auc_score(y_te, y_prob),
            'AUPRC': average_precision_score(y_te, y_prob),
        }
        print(f'  [{name}|{task}] AUROC={baseline_results[name][task]["AUROC"]:.3f}')

# Summary table
rows = []
for name, task_dict in baseline_results.items():
    aurocs = [v['AUROC'] for v in task_dict.values()]
    auprcs = [v['AUPRC'] for v in task_dict.values()]
    rows.append({'Model': name,
                 'Mean AUROC': f'{np.mean(aurocs):.3f} ± {np.std(aurocs):.3f}',
                 'Mean AUPRC': f'{np.mean(auprcs):.3f} ± {np.std(auprcs):.3f}'})

# Add our model
our_aurocs = [results[t]['ci']['AUROC']['point'] for t in results]
our_auprcs = [results[t]['ci']['AUPRC']['point'] for t in results]
rows.append({'Model': 'Ours (Full)',
             'Mean AUROC': f'{np.mean(our_aurocs):.3f} ± {np.std(our_aurocs):.3f}',
             'Mean AUPRC': f'{np.mean(our_auprcs):.3f} ± {np.std(our_auprcs):.3f}'})

bl_table = pd.DataFrame(rows).set_index('Model')
print('\nTABLE 4 — Baseline comparison (mean across 6 endpoints)')
print(bl_table.to_string())
bl_table.to_csv(RESULT_DIR / 'table4_baseline_comparison.csv')

---
## 7 — AD-vs-Performance (Tanimoto-Stratified)

Stratify test compounds by their max Tanimoto similarity to the training set (ECFP4).  
Compare AUROC in low-similarity (out-of-domain) vs high-similarity (in-domain) bins.  
This is the AD analysis promised in Opsi A: data-only diagnostics of applicability domain.

In [ ]:
from rdkit.DataStructs import BulkTanimotoSimilarity

def max_tanimoto_to_train(test_smiles, train_smiles):
    """For each test SMILES, max Tanimoto similarity to any train SMILES (ECFP4)."""
    fps_tr = []
    for smi in train_smiles:
        mol = Chem.MolFromSmiles(str(smi))
        if mol: fps_tr.append(AllChem.GetMorganFingerprintAsBitVect(mol, 2, 2048))

    max_sims = []
    for smi in test_smiles:
        mol = Chem.MolFromSmiles(str(smi))
        if mol is None:
            max_sims.append(0.0)
            continue
        fp  = AllChem.GetMorganFingerprintAsBitVect(mol, 2, 2048)
        sims = BulkTanimotoSimilarity(fp, fps_tr)
        max_sims.append(max(sims) if sims else 0.0)
    return np.array(max_sims)


BINS = [(0.0, 0.4), (0.4, 0.6), (0.6, 1.01)]  # low, mid, high similarity
BIN_LABELS = ['Out-of-domain (<0.4)', 'Mid (0.4–0.6)', 'In-domain (>0.6)']
ad_rows = []

smi_col = 'canonical_smiles'
for task in TASKS:
    tr_df = pd.read_excel(SPLIT_ROOT / task / 'train.xlsx')
    te_df = pd.read_excel(SPLIT_ROOT / task / 'test.xlsx')
    sc    = smi_col if smi_col in tr_df.columns else 'Smiles'

    if task not in results: continue
    yt = results[task]['y_true']
    yp = results[task]['y_prob']

    test_smiles_task  = te_df[sc].tolist()[:len(yt)]
    train_smiles_task = tr_df[sc].tolist()
    max_sim = max_tanimoto_to_train(test_smiles_task, train_smiles_task)

    for (lo, hi), label in zip(BINS, BIN_LABELS):
        idx = np.where((max_sim >= lo) & (max_sim < hi))[0]
        if len(idx) < 10 or len(np.unique(yt[idx])) < 2:
            continue
        auroc = roc_auc_score(yt[idx], yp[idx])
        ad_rows.append({
            'Endpoint': SHORT[task], 'Bin': label, 'n': len(idx), 'AUROC': auroc
        })
        print(f'  [{SHORT[task]}|{label}] n={len(idx)} | AUROC={auroc:.3f}')

ad_df = pd.DataFrame(ad_rows)
ad_df.to_csv(RESULT_DIR / 'ad_vs_performance.csv', index=False)

# Plot
if not ad_df.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(data=ad_df, x='Endpoint', y='AUROC', hue='Bin',
                palette=['#E8593C','#F5A623','#3B8BD4'], ax=ax)
    ax.axhline(0.5, color='k', ls='--', lw=0.8, alpha=0.5)
    ax.set_ylim(0.4, 1.0)
    ax.set_title('AUROC by Applicability Domain (Tanimoto similarity bin)', fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)
    ax.legend(title='Similarity bin', loc='lower right')
    plt.tight_layout()
    plt.savefig(RESULT_DIR / 'ad_vs_performance.png', dpi=200, bbox_inches='tight')
    plt.show()

---
## ✅ Notebook 06 Complete

**Outputs** (`results/evaluation/`):
```
table3_main_performance.csv      ← AUROC/AUPRC/MCC/BalAcc/F1/Sens/Spec + 95% CI
table4_baseline_comparison.csv   ← RF / SVM / XGBoost vs ours
cv_scaffold_vs_random.csv        ← scaffold vs random CV comparison
ad_vs_performance.csv            ← AUROC by Tanimoto similarity bin
figure3_roc_pr_curves.png/pdf    ← 6-panel ROC + PR
ad_vs_performance.png
```

➡️ **Next:** [07_explainable_ai.ipynb](07_explainable_ai.ipynb)